# 전처리

In [1]:
import pandas as pd
import glob
import os

# csv 파일 폴더 경로
CSV_FOLDER = './csv 파일'

# *_menu.csv 전부 탐색
files = sorted(glob.glob(os.path.join(CSV_FOLDER, '*_menu.csv')))
print(f"발견된 파일 수: {len(files)}개")
for f in files:
    print(f"  - {os.path.basename(f)}")

발견된 파일 수: 30개
  - amasvin_menu.csv
  - baekuk_menu.csv
  - banapresso_menu.csv
  - blushaak_menu.csv
  - bombom_menu.csv
  - caffebene_menu.csv
  - caffeine_menu.csv
  - coffeebay_menu.csv
  - compose_coffee_menu.csv
  - dalcu_menu.csv
  - dessert39_menu.csv
  - ediya_menu.csv
  - gongcha_menu.csv
  - hasamdong_menu.csv
  - hiocoffee_menu.csv
  - hollys_menu.csv
  - mammoth_menu.csv
  - mega_coffee_menu.csv
  - oozy_menu.csv
  - paikdabang_menu.csv
  - palgong_menu.csv
  - pascucci_menu.csv
  - paulbassett_menu.csv
  - tenpercent_menu.csv
  - theliter_menu.csv
  - theventi_menu.csv
  - tomntoms_menu.csv
  - twosome_menu.csv
  - vanada_menu.csv
  - yogerpresso_menu.csv


In [2]:
def load_and_normalize(path):
    df = pd.read_csv(path, encoding='utf-8-sig')
    
    # 컬럼명 소문자 정리
    df.columns = df.columns.str.strip().str.lower()
    
    # 컬럼명 유연하게 매핑 (파일마다 다를 수 있어서)
    col_map = {
        'menu': 'name', 'menu_name': 'name', '메뉴명': 'name', '메뉴': 'name',
        'brand_name': 'brand', '브랜드': 'brand',
        'temperature': 'temp', '온도': 'temp',
        'cat': 'category', '카테고리': 'category',
        'iscoffee': 'is_coffee', 'coffee': 'is_coffee',
    }
    df = df.rename(columns=col_map)
    
    # brand 컬럼 없으면 파일명에서 추출
    if 'brand' not in df.columns:
        brand_guess = os.path.basename(path).replace('_menu.csv', '').replace('_', ' ')
        df['brand'] = brand_guess
    
    # 필수 컬럼 없으면 빈값으로 생성
    for col in ['brand', 'name', 'temp', 'category', 'is_coffee']:
        if col not in df.columns:
            df[col] = None
    
    return df[['brand', 'name', 'temp', 'category', 'is_coffee']].copy()

# 전체 로드
dfs = []
for path in files:
    df = load_and_normalize(path)
    print(f"{os.path.basename(path):<35} {len(df)}행  |  컬럼: {list(df.columns)}")
    dfs.append(df)

# 합치기
combined = pd.concat(dfs, ignore_index=True)
print(f"\n합친 후 총 행수: {len(combined)}행")

amasvin_menu.csv                    81행  |  컬럼: ['brand', 'name', 'temp', 'category', 'is_coffee']
baekuk_menu.csv                     75행  |  컬럼: ['brand', 'name', 'temp', 'category', 'is_coffee']
banapresso_menu.csv                 187행  |  컬럼: ['brand', 'name', 'temp', 'category', 'is_coffee']
blushaak_menu.csv                   115행  |  컬럼: ['brand', 'name', 'temp', 'category', 'is_coffee']
bombom_menu.csv                     130행  |  컬럼: ['brand', 'name', 'temp', 'category', 'is_coffee']
caffebene_menu.csv                  74행  |  컬럼: ['brand', 'name', 'temp', 'category', 'is_coffee']
caffeine_menu.csv                   62행  |  컬럼: ['brand', 'name', 'temp', 'category', 'is_coffee']
coffeebay_menu.csv                  102행  |  컬럼: ['brand', 'name', 'temp', 'category', 'is_coffee']
compose_coffee_menu.csv             144행  |  컬럼: ['brand', 'name', 'temp', 'category', 'is_coffee']
dalcu_menu.csv                      88행  |  컬럼: ['brand', 'name', 'temp', 'category', 'is_coffee']
desse

In [3]:
# 결측값 확인
print("=== 결측값 현황 ===")
print(combined.isnull().sum())

print(f"\n=== 중복 제거 전: {len(combined)}행 ===")

# name이 비어있는 행 제거
combined = combined.dropna(subset=['name'])
combined = combined[combined['name'].str.strip() != '']

# brand + name 기준 중복 제거
combined = combined.drop_duplicates(subset=['brand', 'name'])

print(f"=== 중복 제거 후: {len(combined)}행 ===")
print(f"제거된 행: {4265 - len(combined)}개")

# 브랜드별 메뉴 수 확인
print("\n=== 브랜드별 메뉴 수 ===")
print(combined['brand'].value_counts().to_string())

=== 결측값 현황 ===
brand           0
name            0
temp         1766
category        0
is_coffee       0
dtype: int64

=== 중복 제거 전: 4265행 ===
=== 중복 제거 후: 3600행 ===
제거된 행: 665개

=== 브랜드별 메뉴 수 ===
brand
빽다방       269
요거프레소     260
이디야       251
바나프레소     187
커피에반하다    181
우지커피      173
하이오커피     160
컴포즈커피     144
디저트39     134
하삼동커피     129
카페봄봄      122
블루샥       115
매머드커피     113
텐퍼센트커피    109
메가커피      108
폴바셋       106
커피베이       99
팔공티        88
더벤티        87
파스쿠찌       83
더리터        82
투썸플레이스     76
백억커피       75
달리는커피      74
아마스빈       71
할리스        70
탐앤탐스       67
카페인중독      60
카페베네       56
공차         51


In [4]:
ICE_KW = ['아이스', 'iced', 'ice', '냉', '스무디', '쉐이크', '에이드',
          '주스', '소다', '프라페', '블렌디드', '프라푸치노', '슬러시']
HOT_KW = ['따뜻', '따뜻한', 'hot', '핫', '뜨거운']
ALWAYS_COLD_CAT = ['주스', '에이드', '스무디', '쉐이크', '요거트',
                   '프라페', '블렌디드', '생과일', '착즙']

def infer_temp(row):
    t = str(row.get('temp', '')).strip().upper()
    name = str(row['name']).lower()
    cat  = str(row.get('category', '')).lower()

    # 이미 명시된 경우
    if t in ['ICE', 'HOT']:
        return t
    if t in ['HOT/ICE', 'ICE/HOT', 'BOTH', 'HOT&ICE']:
        return 'BOTH'

    # 카테고리로 추론
    if any(k in cat for k in ALWAYS_COLD_CAT):
        return 'ICE'

    # 메뉴명으로 추론
    if any(k in name for k in ICE_KW):
        return 'ICE'
    if any(k in name for k in HOT_KW):
        return 'HOT'

    # 콜드브루 원액
    if '원액' in name or '콜드브루' in cat:
        return 'COLD_BREW'

    return 'BOTH'  # 추론 불가 → HOT/ICE 둘 다 가능

combined['temp'] = combined.apply(infer_temp, axis=1)

print("=== temp 분포 ===")
print(combined['temp'].value_counts())

print("\n=== BOTH 샘플 (온도 미지정) ===")
print(combined[combined['temp'] == 'BOTH'][['brand', 'name', 'category']].head(10).to_string())

=== temp 분포 ===
temp
ICE          1820
BOTH         1336
HOT           433
COLD_BREW      11
Name: count, dtype: int64

=== BOTH 샘플 (온도 미지정) ===
   brand        name category
0   아마스빈       아메리카노       커피
2   아마스빈   바닐라 아메리카노       커피
3   아마스빈  헤이즐넛 아메리카노       커피
4   아마스빈        카페라떼       커피
5   아마스빈      바닐라 라떼       커피
6   아마스빈     헤이즐넛 라떼       커피
7   아마스빈    카라멜 마끼아또       커피
8   아마스빈       카페 모카       커피
9   아마스빈       연유 라떼       커피
10  아마스빈    얼그레이 밀크티   밀크티/라떼


In [5]:
COFFEE_CORE_KW = ['아메리카노', '에스프레소', '카푸치노', '마끼아또', '마키아토',
                  '콜드브루', '카페라떼', '카페 라떼', '슈페너',
                  'americano', 'espresso', 'coffee']

NOT_COFFEE_CAT_KW = ['논커피', '음료', '주스', '에이드', '스무디', '쉐이크',
                     '요거트', '빙수', '디저트', '티', '허브',
                     'ade', 'tea', 'juice', 'smoothie', 'shake', 'dessert']

NOT_COFFEE_NAME_KW = ['과일', '주스', '에이드', '소다', '요거트', '스무디',
                      '쉐이크', '밀크티', '버블티', '녹차', '흑당', '아이스티',
                      '페퍼민트', '캐모마일', '루이보스', '허브티', '레모네이드',
                      '생과일', '착즙', '히비스커스', '대추차', '보리차',
                      '밀크컵', '빙수', '케이크', '말차라떼', '말차 라떼',
                      '로얄밀크티', '로얄 밀크티']

def fix_is_coffee(row):
    name = str(row['name']).lower()
    cat  = str(row.get('category', '')).lower()

    # 1순위: 카테고리에 논커피 명시 → 무조건 False
    if any(k in cat for k in NOT_COFFEE_CAT_KW):
        return False

    # 2순위: 커피 핵심 키워드 있으면 True
    if any(k in name for k in COFFEE_CORE_KW):
        return True

    # 3순위: 메뉴명에 논커피 키워드 있으면 False
    if any(k in name for k in NOT_COFFEE_NAME_KW):
        return False

    # 추론 불가 시 원본값 유지
    v = row['is_coffee']
    if pd.isna(v):
        return False
    return bool(v)

before = combined['is_coffee'].copy()
combined['is_coffee'] = combined.apply(fix_is_coffee, axis=1)

changed = combined[before != combined['is_coffee']][['brand', 'name', 'category', 'is_coffee']]
print(f"=== is_coffee 수정된 행: {len(changed)}개 ===")

print("\n=== is_coffee 분포 ===")
print(combined['is_coffee'].value_counts())

print("\n=== True 샘플 15개 ===")
print(combined[combined['is_coffee'] == True][['brand', 'name', 'category']].sample(15).to_string())

print("\n=== False 샘플 15개 ===")
print(combined[combined['is_coffee'] == False][['brand', 'name', 'category']].sample(15).to_string())

=== is_coffee 수정된 행: 150개 ===

=== is_coffee 분포 ===
is_coffee
False    2607
True      993
Name: count, dtype: int64

=== True 샘플 15개 ===
       brand                   name   category
976    달리는커피                  시나몬모카     Coffee
3228    파스쿠찌          아이스 바닐라빈 골든라떼   커피(ICED)
3688    탐앤탐스                오트 콜드브루   COLDBREW
2755     빽다방   디카페인 코코넛콜드브루라떼(ICED)         커피
231    바나프레소               빅바나 콜드브루        빅바나
2832     빽다방  디카페인 빽사이즈 바닐라라떼(ICED)         커피
349      블루샥               바닐라 빈 라떼     Coffee
3848  커피에반하다                 콜드브루라떼   에스프레소베이스
1791   하삼동커피              콜드브루 돌체라떼  콜드브루/디카페인
194    바나프레소          디카페인 제로슈가 아샷추    저당&제로슈가
1788   하삼동커피                 흑당카페라떼         커피
96      백억커피                   콜드브루       콜드브루
363      블루샥              화이트 아메리카노     Coffee
345      블루샥              나이트 아메리카노     Coffee
2163   매머드커피             디카페인 돌체 라떼         커피

=== False 샘플 15개 ===
       brand                  name          category
2571    우지커피             망고 요거트스무디  

In [6]:
import re

def clean_name(name):
    name = str(name).strip()
    name = re.sub(r'\s*[\(\（][A-Za-z\s]{1,10}[\)\）]\s*', ' ', name)
    name = re.sub(r'\s*[\(\（][LMS][\)\）]\s*', ' ', name)
    name = re.sub(r'\s+', ' ', name).strip()
    return name

before_names = combined['name'].copy()
combined['name'] = combined['name'].apply(clean_name)

changed_names = combined[before_names != combined['name']][['brand', 'name']]
print(f"=== 메뉴명 수정된 행: {len(changed_names)}개 ===")

# 메뉴명 정제 후 새로 생긴 중복 제거
before_dedup = len(combined)
combined = combined.drop_duplicates(subset=['brand', 'name'])
print(f"=== 정제 후 중복 제거: {before_dedup - len(combined)}개 제거 → {len(combined)}행 ===")

# 망고라떼 False 처리
NOT_COFFEE_NAME_KW = ['과일', '주스', '에이드', '소다', '요거트', '스무디',
                      '쉐이크', '밀크티', '버블티', '녹차', '흑당', '아이스티',
                      '페퍼민트', '캐모마일', '루이보스', '허브티', '레모네이드',
                      '생과일', '착즙', '히비스커스', '대추차', '보리차',
                      '밀크컵', '빙수', '케이크', '말차라떼', '말차 라떼',
                      '로얄밀크티', '로얄 밀크티', '망고라떼', '망고 라떼']

combined.loc[
    combined['name'].str.lower().apply(lambda x: any(k in x for k in NOT_COFFEE_NAME_KW)),
    'is_coffee'
] = False

print("\n=== 최종 is_coffee 분포 ===")
print(combined['is_coffee'].value_counts())
print(f"\n=== 최종 행수: {len(combined)} ===")

# 망고라떼 확인
print("\n=== 망고라떼 확인 ===")
print(combined[combined['name'].str.contains('망고라떼|망고 라떼', na=False)][['brand', 'name', 'is_coffee']].to_string())

=== 메뉴명 수정된 행: 629개 ===
=== 정제 후 중복 제거: 206개 제거 → 3394행 ===

=== 최종 is_coffee 분포 ===
is_coffee
False    2482
True      912
Name: count, dtype: int64

=== 최종 행수: 3394 ===

=== 망고라떼 확인 ===
       brand        name  is_coffee
378      블루샥       망고 라떼      False
452      블루샥  1리터보틀 망고라떼      False
905    컴포즈커피        망고라떼      False
1156   디저트39     저당 망고라떼      False
1257   디저트39        망고라떼      False
1763   하삼동커피   말차구름 망고라떼      False
3861  커피에반하다    딸기 망고 라떼      False


## 추가 정리

In [7]:
import re

# ── 삭제할 행 처리 ──────────────────────────────────────────

# 요거프레소 토핑 행 삭제
before = len(combined)
combined = combined[~combined['name'].str.contains(r'\[토핑', na=False)]
print(f"요거프레소 토핑 삭제: {before - len(combined)}행 제거")

# 텐퍼센트 신메뉴 카테고리 삭제 (묶음 상품)
before = len(combined)
combined = combined[~(
    (combined['brand'] == '텐퍼센트커피') & 
    (combined['category'] == '신메뉴')
)]
print(f"텐퍼센트 신메뉴 삭제: {before - len(combined)}행 제거")

# ── 메뉴명 노이즈 제거 ──────────────────────────────────────

def clean_name_extra(name):
    name = str(name).strip()
    
    # HTML 태그 제거: <br>, <br/> → 공백
    name = re.sub(r'<br\s*/?>', ' ', name, flags=re.IGNORECASE)
    
    # NEW) 제거
    name = re.sub(r'NEW\)\s*', '', name, flags=re.IGNORECASE)
    
    # [인기옵션], [시즌], [리뉴얼] 등 대괄호 태그 제거
    name = re.sub(r'\[[^\]]{1,10}\]\s*', '', name)
    
    # 연속 공백 정리
    name = re.sub(r'\s+', ' ', name).strip()
    
    return name

before_names = combined['name'].copy()
combined['name'] = combined['name'].apply(clean_name_extra)

changed = combined[before_names != combined['name']][['brand', 'name']]
print(f"\n메뉴명 노이즈 수정: {len(changed)}행")
print(changed.head(20).to_string())

# 정제 후 새로 생긴 중복 제거
before = len(combined)
combined = combined.drop_duplicates(subset=['brand', 'name'])
print(f"\n정제 후 중복 제거: {before - len(combined)}개 → {len(combined)}행")

print(f"\n✅ 최종 행수: {len(combined)}행")

요거프레소 토핑 삭제: 33행 제거
텐퍼센트 신메뉴 삭제: 48행 제거

메뉴명 노이즈 수정: 32행
      brand                name
720   카페인중독              천혜향 카노
721   카페인중독          코코 천혜향 스무디
722   카페인중독            밀크카우 스무디
723   카페인중독        제주 사려니 숲길 라떼
990   달리는커피        달리터 딸기 요거트라떼
991   달리는커피     달리터 납작복숭아 요거트라떼
993   달리는커피      달리터 믹스베리 요거트라떼
994   달리는커피       달리터 구아바 요거트라떼
1030  달리는커피        달리터 딸기파인 에이드
1031  달리는커피      달리터 딸기복숭아 아이스티
1032  달리는커피      구아바캐모마일 저당블렌딩티
1033  달리는커피       베리얼그레이 저당블렌딩티
1034  달리는커피       애플얼그레이 저당블렌딩티
1195  디저트39             디카페인 원두
2059    할리스         블랙아리아 아메리카노
2060    할리스           블랙아리아 딥라떼
2062    할리스           디카페인 콜드브루
2063    할리스        디카페인 콜드브루 라떼
2064    할리스  디카페인 콜드브루 바닐라 딜라이트
2086    할리스         핑크 파인 캐모마일티

정제 후 중복 제거: 0개 → 3313행

✅ 최종 행수: 3313행


In [10]:
# 컬럼 순서 정리
combined = combined[['brand', 'name', 'temp', 'category', 'is_coffee']]

# brand, category, name 기준으로 정렬
combined = combined.sort_values(['brand', 'category', 'name']).reset_index(drop=True)

# output 폴더 없으면 생성
os.makedirs('./output', exist_ok=True)

# 저장
combined.to_csv('./output/cafe_menu_processed.csv', index=False, encoding='utf-8-sig')

# 최종 리포트
print("=" * 40)
print("       최종 전처리 완료 리포트")
print("=" * 40)
print(f"총 메뉴 수     : {len(combined)}개")
print(f"브랜드 수      : {combined['brand'].nunique()}개")
print(f"\n[temp 분포]")
for k, v in combined['temp'].value_counts().items():
    print(f"  {k:<12}: {v}개")
print(f"\n[is_coffee 분포]")
for k, v in combined['is_coffee'].value_counts().items():
    label = '커피' if k else '논커피'
    print(f"  {label:<10}: {v}개")
print(f"\n[브랜드별 메뉴 수]")
print(combined['brand'].value_counts().to_string())
print(f"\n✅ 저장 완료 → ./output/cafe_menu_processed.csv")

       최종 전처리 완료 리포트
총 메뉴 수     : 3313개
브랜드 수      : 30개

[temp 분포]
  ICE         : 1644개
  BOTH        : 1269개
  HOT         : 389개
  COLD_BREW   : 11개

[is_coffee 분포]
  논커피       : 2403개
  커피        : 910개

[브랜드별 메뉴 수]
brand
요거프레소     225
빽다방       191
바나프레소     187
커피에반하다    181
우지커피      173
하이오커피     160
컴포즈커피     144
디저트39     134
하삼동커피     129
이디야       128
카페봄봄      122
블루샥       115
매머드커피     113
폴바셋       106
메가커피      105
커피베이       99
팔공티        88
더벤티        87
파스쿠찌       83
더리터        82
투썸플레이스     76
백억커피       75
달리는커피      74
아마스빈       71
할리스        70
탐앤탐스       67
텐퍼센트커피     61
카페인중독      60
카페베네       56
공차         51

✅ 저장 완료 → ./output/cafe_menu_processed.csv


In [12]:
import re

# ① 커피베이 (HOT/ICE) 괄호 제거 + temp BOTH로 변경
mask = combined['name'].str.contains(r'\(HOT/ICE\)', na=False)
combined.loc[mask, 'name'] = combined.loc[mask, 'name'].str.replace(r'\(HOT/ICE\)', '', regex=True).str.strip()
combined.loc[mask, 'temp'] = 'BOTH'
print(f"커피베이 HOT/ICE 처리: {mask.sum()}개")

# ② 할리스 </br> 태그 제거
mask2 = combined['name'].str.contains(r'</br>', na=False)
combined.loc[mask2, 'name'] = combined.loc[mask2, 'name'].str.replace(r'</br>', ' ', regex=True)
combined.loc[mask2, 'name'] = combined.loc[mask2, 'name'].apply(lambda x: re.sub(r'\s+', ' ', x).strip())
print(f"할리스 </br> 처리: {mask2.sum()}개")

# ③ 디저트39 굿즈 행 삭제
before = len(combined)
combined = combined[~((combined['brand'] == '디저트39') & (combined['name'] == '슈의 키캡&키링'))]
print(f"디저트39 굿즈 삭제: {before - len(combined)}개")

# 결과 확인
print(f"\n최종 행수: {len(combined)}행")

# 저장
combined = combined[['brand', 'name', 'temp', 'category', 'is_coffee']]
combined = combined.sort_values(['brand', 'category', 'name']).reset_index(drop=True)
os.makedirs('./output', exist_ok=True)
combined.to_csv('./output/cafe_menu_processed.csv', index=False, encoding='utf-8-sig')
print("저장 완료")

커피베이 HOT/ICE 처리: 3개
할리스 </br> 처리: 4개
디저트39 굿즈 삭제: 1개

최종 행수: 3312행
저장 완료


In [16]:
# 순수 음식류 삭제 (brand + name 조합으로 정확하게)
food_to_delete = [
    # 요거프레소 음식류
    ('요거프레소', '다크초코칩 크로플'), ('요거프레소', '딸기 크로플'),
    ('요거프레소', '망고 크로플'), ('요거프레소', '블루베리 크로플'),
    ('요거프레소', '아몬드초코카라멜 크로플'), ('요거프레소', '카라멜 크로플'),
    ('요거프레소', '크로플'), ('요거프레소', '쫀득베이글 블루베리'),
    ('요거프레소', '쫀득베이글 어니언'), ('요거프레소', '쫀득베이글 플레인'),
    ('요거프레소', '칠리앤치즈 핫도그'), ('요거프레소', '콘어니언 핫도그'),
    ('요거프레소', '플레인 츄러스'), ('요거프레소', '크로크무슈'),
    ('요거프레소', '허니브래드'),
    ('요거프레소', '마이모토 크리스피 웨이퍼(코코아)'),
    ('요거프레소', '마이모토 크리스피 웨이퍼(티라미수)'),
    # 디저트39 베이글
    ('디저트39', '베이글 온 더 아메리카노(어니언바질)'),
    ('디저트39', '베이글 온 더 아메리카노(치즈)'),
    # 컴포즈커피 떡볶이
    ('컴포즈커피', '쫄깃 분모자 떡볶이'),
]

before = len(combined)
for brand, name in food_to_delete:
    combined = combined[~((combined['brand'] == brand) & (combined['name'] == name))]

print(f"삭제된 행: {before - len(combined)}개")
print(f"최종 행수: {len(combined)}행")

combined.to_csv('./output/cafe_menu_processed.csv', index=False, encoding='utf-8-sig')
print("저장 완료")

삭제된 행: 20개
최종 행수: 3272행
저장 완료
